In [1]:
# Fix TF/protobuf runtime crash seen as:
# AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
# In some Kaggle images, TensorFlow + protobuf 4/5/6 can fail unless we force
# the pure-Python protobuf implementation BEFORE importing tensorflow.
import os

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
from PIL import Image
import pydicom
import matplotlib.pyplot as plt
import pylab
import cv2
from tensorflow.keras.utils import Sequence
from tqdm import tqdm

import tensorflow as tf
import tensorflow.keras.layers as L
import tensorflow.keras.models as M
import tensorflow.keras.backend as K
import tensorflow.keras.regularizers as R
from sklearn.metrics import mean_absolute_error

np.random.seed(24)
tf.random.set_seed(24)



2026-01-23 21:52:13.063568: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769205133.078262      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769205133.082708      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
comp_dir = "../input/osic-pulmonary-fibrosis-progression"

train_data = pd.read_csv(os.path.join(comp_dir, "train.csv"))
sub = pd.read_csv(os.path.join(comp_dir, "sample_submission.csv"))
test_data = pd.read_csv(os.path.join(comp_dir, "test.csv"))
train_data.drop_duplicates(keep=False, inplace=True, subset=["Patient", "Weeks"])



In [3]:
train_data.head()



,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus
0,ID00133637202223847701934,-2,3195,92.856312,83,Male,Never smoked
1,ID00133637202223847701934,2,3203,93.088817,83,Male,Never smoked
2,ID00133637202223847701934,4,3097,90.008138,83,Male,Never smoked
3,ID00133637202223847701934,6,3171,92.158800,83,Male,Never smoked
4,ID00133637202223847701934,8,3115,90.531272,83,Male,Never smoked


In [4]:
test_data.head()



,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus
0,ID00014637202177757139317,0,3807,90.076661,56,Male,Ex-smoker
1,ID00019637202178323708467,13,2100,92.858722,83,Female,Ex-smoker
2,ID00047637202184938901501,2,3313,89.929425,68,Male,Ex-smoker
3,ID00082637202201836229724,19,2918,99.794802,49,Female,Currently smokes
4,ID00126637202218610655908,18,2375,59.375000,78,Male,Ex-smoker


In [5]:
sub.head()



,Patient_Week,FVC,Confidence
0,ID00126637202218610655908_-3,2000,100
1,ID00126637202218610655908_-2,2000,100
2,ID00126637202218610655908_-1,2000,100
3,ID00126637202218610655908_0,2000,100
4,ID00126637202218610655908_1,2000,100


In [6]:
train_data_u = train_data
train_data_u = train_data_u.drop_duplicates(subset=["Patient"])
train_data_u = train_data_u.rename(
    columns={"Weeks": "Base_Week", "FVC": "Base_FVC", "Percent": "Base_Percent"}
)
train_data_u["Typical_FVC"] = (
    train_data_u.Base_FVC.values / train_data_u.Base_Percent.values
) * 100
train_data = train_data.merge(
    train_data_u.drop(["Age", "Sex", "SmokingStatus"], axis=1), on="Patient", how="left"
)



In [7]:
sub["Patient"] = sub["Patient_Week"].apply(lambda x: x.split("_")[0])
sub["Weeks"] = sub["Patient_Week"].apply(lambda x: int(x.split("_")[-1]))
sub = sub.drop(["Confidence"], axis=1)
sub = sub[["Patient", "Weeks", "Patient_Week"]]



In [8]:
test_data = test_data.rename(
    columns={"Weeks": "Base_Week", "FVC": "Base_FVC", "Percent": "Base_Percent"}
)
test_data["Typical_FVC"] = (
    test_data.Base_FVC.values / test_data.Base_Percent.values
) * 100

# Merge baseline features onto each requested Patient_Week row.
# Keep sub['Weeks'] as the prediction week feature.
sub = sub.merge(test_data, how="left", on="Patient", suffixes=("", "_y"))



In [9]:
train_data["Type"] = "train"
sub["Type"] = "test"



In [10]:
# pandas 2.x removed DataFrame.append; use concat to preserve behavior.
data = pd.concat([train_data, sub], ignore_index=True, sort=False)



In [11]:
prediction_col = ["FVC"]
Continuos_cols = [
    "Weeks",
    "Base_Week",
    "Base_FVC",
    "Typical_FVC",
    "Age",
    "Percent",
    "Base_Percent",
]

# Keep for compatibility with the original notebook intent (not used later)
Categorical_cols = ["Sex", "SmokingStatus"]



In [12]:
from sklearn.preprocessing import MinMaxScaler



In [13]:
# Ensure required numeric columns exist (Percent exists in train, but not in sub rows after merge).
# In test (sub rows), Percent is unknown at future weeks; use Base_Percent as a stable proxy.
if "Percent" not in data.columns:
    data["Percent"] = np.nan
data.loc[data["Type"] == "test", "Percent"] = data.loc[
    data["Type"] == "test", "Percent"
].fillna(data.loc[data["Type"] == "test", "Base_Percent"])
# In case any remaining missing values exist, fill from patient baseline.
data["Percent"] = data["Percent"].fillna(data["Base_Percent"])

# Some rows may have missing Age/Sex/SmokingStatus after merge; fill conservatively.
data["Age"] = data["Age"].fillna(data["Age"].median())
data["Sex"] = data["Sex"].fillna("Male")
data["SmokingStatus"] = data["SmokingStatus"].fillna("Ex-smoker")

scaler = MinMaxScaler()
conti = scaler.fit_transform(data[Continuos_cols])
data[Continuos_cols] = conti



In [14]:
print(
    np.mean(train_data_u.query("SmokingStatus == 'Never smoked'").Base_Percent.values)
)
print(
    np.mean(
        train_data_u.query("SmokingStatus == 'Currently smokes'").Base_Percent.values
    )
)
print(np.mean(train_data_u.query("SmokingStatus == 'Ex-smoker'").Base_Percent.values))
print(np.mean(train_data_u.query("Sex == 'Male'").Base_Percent.values))
print(np.mean(train_data_u.query("Sex == 'Female'").Base_Percent.values))



78.68346533571169
100.4693934999359
78.08689959858515
77.95708401998029
84.33313954809648


In [15]:
# Encode categoricals to numeric as in original logic (minimal change; keep same mapping).
# Use .loc assignments to avoid chained assignment pitfalls.
sex_vals = data["Sex"].values
for i in range(len(sex_vals)):
    if sex_vals[i] == "Male":
        sex_vals[i] = 0
    else:
        sex_vals[i] = 1
data["Sex"] = sex_vals.astype(np.float32)

smoke_vals = data["SmokingStatus"].values
for i in range(len(smoke_vals)):
    if smoke_vals[i] == "Ex-smoker":
        smoke_vals[i] = 0
    elif smoke_vals[i] == "Never smoked":
        smoke_vals[i] = 1
    else:
        smoke_vals[i] = 2
data["SmokingStatus"] = smoke_vals.astype(np.float32)



In [16]:
data



/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus,Base_Week,Base_FVC,Base_Percent,Typical_FVC,Type,Patient_Week
0,ID00133637202223847701934,0.021739,3195.0,0.514846,0.871795,0.0,1.0,0.035714,0.404903,0.451429,0.554072,train,NaN
1,ID00133637202223847701934,0.050725,3203.0,0.516717,0.871795,0.0,1.0,0.035714,0.404903,0.451429,0.554072,train,NaN
2,ID00133637202223847701934,0.065217,3097.0,0.491926,0.871795,0.0,1.0,0.035714,0.404903,0.451429,0.554072,train,NaN
3,ID00133637202223847701934,0.079710,3171.0,0.509233,0.871795,0.0,1.0,0.035714,0.404903,0.451429,0.554072,train,NaN
4,ID00133637202223847701934,0.094203,3115.0,0.496136,0.871795,0.0,1.0,0.035714,0.404903,0.451429,0.554072,train,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3283,ID00047637202184938901501,0.746377,NaN,0.491293,0.487179,0.0,0.0,0.083333,0.426820,0.424739,0.627230,test,ID00047637202184938901501_98
3284,ID00047637202184938901501,0.753623,NaN,0.491293,0.487179,0.0,0.0,0.083333,0.426820,0.424739,0.627230,test,ID00047637202184938901501_99
3285,ID00047637202184938901501,0.760870,NaN,0.491293,0.487179,0.0,0.0,0.083333,0.426820,0.424739,0.627230,test,ID00047637202184938901501_100
3286,ID00047637202184938901501,0.768116,NaN,0.491293,0.487179,0.0,0.0,0.083333,0.426820,0.424739,0.627230,test,ID00047637202184938901501_101


In [17]:
x_cols = [
    "Weeks",
    "Base_Week",
    "Sex",
    "SmokingStatus",
    "Age",
    "Base_Percent",
    "Typical_FVC",
]



In [18]:
data



/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus,Base_Week,Base_FVC,Base_Percent,Typical_FVC,Type,Patient_Week
0,ID00133637202223847701934,0.021739,3195.0,0.514846,0.871795,0.0,1.0,0.035714,0.404903,0.451429,0.554072,train,NaN
1,ID00133637202223847701934,0.050725,3203.0,0.516717,0.871795,0.0,1.0,0.035714,0.404903,0.451429,0.554072,train,NaN
2,ID00133637202223847701934,0.065217,3097.0,0.491926,0.871795,0.0,1.0,0.035714,0.404903,0.451429,0.554072,train,NaN
3,ID00133637202223847701934,0.079710,3171.0,0.509233,0.871795,0.0,1.0,0.035714,0.404903,0.451429,0.554072,train,NaN
4,ID00133637202223847701934,0.094203,3115.0,0.496136,0.871795,0.0,1.0,0.035714,0.404903,0.451429,0.554072,train,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3283,ID00047637202184938901501,0.746377,NaN,0.491293,0.487179,0.0,0.0,0.083333,0.426820,0.424739,0.627230,test,ID00047637202184938901501_98
3284,ID00047637202184938901501,0.753623,NaN,0.491293,0.487179,0.0,0.0,0.083333,0.426820,0.424739,0.627230,test,ID00047637202184938901501_99
3285,ID00047637202184938901501,0.760870,NaN,0.491293,0.487179,0.0,0.0,0.083333,0.426820,0.424739,0.627230,test,ID00047637202184938901501_100
3286,ID00047637202184938901501,0.768116,NaN,0.491293,0.487179,0.0,0.0,0.083333,0.426820,0.424739,0.627230,test,ID00047637202184938901501_101


In [19]:
x_train = data[x_cols].loc[data["Type"] == "train"].values
y_train = data[prediction_col].loc[data["Type"] == "train"].values
x_test = data[x_cols].loc[data["Type"] == "test"].values



In [20]:
x_train = x_train.astype(np.float32)
y_train = y_train.astype(np.float32)
x_test = x_test.astype(np.float32)



In [21]:
x_train.shape, y_train.shape, x_test.shape



((1380, 7), (1380, 1), (1908, 7))

In [22]:
type(x_train), type(y_train), type(x_test)



(numpy.ndarray, numpy.ndarray, numpy.ndarray)

In [23]:
C1, C2 = tf.constant(70, dtype="float32"), tf.constant(1000, dtype="float32")


def score(y_true, y_pred):
    tf.dtypes.cast(y_true, tf.float32)
    tf.dtypes.cast(y_pred, tf.float32)
    sigma = y_pred[:, 2] - y_pred[:, 0]
    fvc_pred = y_pred[:, 1]
    sigma_clip = tf.maximum(sigma, C1)
    delta = tf.abs(y_true[:, 0] - fvc_pred)
    delta = tf.minimum(delta, C2)
    sq2 = tf.sqrt(tf.dtypes.cast(2, dtype=tf.float32))
    metric = (delta / sigma_clip) * sq2 + tf.math.log(sigma_clip * sq2)
    return K.mean(metric)


def qloss(y_true, y_pred):
    qs = [0.2, 0.50, 0.8]
    q = tf.constant(np.array([qs]), dtype=tf.float32)
    e = y_true - y_pred
    v = tf.maximum(q * e, (q - 1) * e)
    return K.mean(v)


def mloss(_lambda):
    def loss(y_true, y_pred):
        return _lambda * qloss(y_true, y_pred) + (1 - _lambda) * score(y_true, y_pred)

    return loss




I0000 00:00:1769205140.644812      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:86:00.0, compute capability: 8.6


In [24]:
def build_model():
    inp = L.Input((7,))
    x = L.Dense(128, activation="relu", kernel_regularizer=R.l2(1e-6))(inp)
    x = L.Dense(128, activation="relu", kernel_regularizer=R.l2(1e-6))(x)
    x = L.Dense(64, activation="relu", kernel_regularizer=R.l2(1e-6))(x)
    o1 = L.Dense(3, activation="linear")(x)
    o2 = L.Dense(3, activation="relu")(x)
    pred1 = L.Lambda(lambda x: (x[0] + (tf.cumsum(x[1], axis=1))))([o1, o2])

    model = M.Model(inputs=inp, outputs=pred1)
    model.compile(
        loss=mloss(0.8),
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-2),
        metrics=[score],
    )
    return model




In [25]:
model = build_model()
model.summary()



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │      1,024 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     16,512 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 3)         │        195 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 3)         │        195 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 3)         │          0 │ dense_3[0][0],    │
│                     │                   │            │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 26,182 (102.27 KB)

 Trainable params: 26,182 (102.27 KB)

 Non-trainable params: 0 (0.00 B)

In [26]:
from sklearn.model_selection import KFold

folds = 5
KF = KFold(n_splits=folds, shuffle=True, random_state=24)



In [27]:
# Notebook magic "%%time" removed to keep this runnable as a .py script in Kaggle.
counter = 0
stopper = tf.keras.callbacks.EarlyStopping(
    monitor="loss", mode="min", patience=200, restore_best_weights=True
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="loss", factor=0.1, patience=50, min_lr=1e-3
)

for tr_idx, val_idx in KF.split(x_train):
    counter += 1
    print(f"############## FOLD {counter} ###############")
    model.fit(
        x_train[tr_idx],
        y_train[tr_idx],
        epochs=800,
        batch_size=256,
        verbose=0,
        validation_data=(x_train[val_idx], y_train[val_idx]),
        callbacks=[reduce_lr],
    )
    print(
        "train",
        model.evaluate(x_train[tr_idx], y_train[tr_idx], verbose=0, batch_size=256),
    )
    print(
        "val",
        model.evaluate(x_train[val_idx], y_train[val_idx], verbose=0, batch_size=256),
    )



############## FOLD 1 ###############


I0000 00:00:1769205142.819241      63 service.cc:148] XLA service 0x7f95a000afb0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769205142.819286      63 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1769205142.970261      63 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1769205148.210875      63 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


train [49.11798858642578, 6.660359859466553]
val [51.49956512451172, 6.587896347045898]
############## FOLD 2 ###############
train [46.165565490722656, 6.601365089416504]
val [54.13953399658203, 6.639521598815918]
############## FOLD 3 ###############
train [45.96234130859375, 6.563174247741699]
val [42.967308044433594, 6.555356025695801]
############## FOLD 4 ###############
train [44.38413619995117, 6.535490989685059]
val [44.42680740356445, 6.605557918548584]
############## FOLD 5 ###############
train [42.65980911254883, 6.492382049560547]
val [48.172969818115234, 6.725543975830078]


In [28]:
model.save("model.h5")



In [29]:
pred = model.predict(x_train, verbose=0)



In [30]:
sigma_opt = mean_absolute_error(y_train, pred[:, 1])
unc = pred[:, 2] - pred[:, 0]
sigma_mean = np.mean(unc)
sigma = pred[:, 2] - pred[:, 0]
print(sigma_opt, sigma_mean, sigma[:10])



127.18374 229.80132 [164.59326 181.92896 190.60791 199.13208 207.64404 228.98413 288.71802
 335.34912 393.68652  84.70654]


In [31]:
pred[:10]



array([[3069.3691, 3156.4888, 3233.9624],
       [3042.6587, 3140.8252, 3224.5876],
       [3029.9893, 3133.6045, 3220.5972],
       [3016.8267, 3125.8333, 3215.9587],
       [3003.5576, 3118.0527, 3211.2017],
       [2971.565 , 3099.631 , 3200.549 ],
       [2880.6343, 3046.6094, 3169.3523],
       [2813.5903, 3009.1487, 3148.9395],
       [2801.776 , 3034.0518, 3195.4624],
       [1629.0117, 1671.7151, 1713.7183]], dtype=float32)

In [32]:
pred = model.predict(x_test, verbose=0)



In [33]:
pred



array([[2464.2808, 2517.3987, 2570.722 ],
       [2452.6768, 2508.671 , 2563.5159],
       [2442.065 , 2500.5513, 2556.837 ],
       ...,
       [2899.1113, 3204.3179, 3408.427 ],
       [2903.6982, 3210.04  , 3414.7424],
       [2908.5935, 3215.9722, 3421.3752]], dtype=float32)

In [34]:
conf = pred[:, 2] - pred[:, 0]
for i in range(len(conf)):
    conf[i] = max(conf[i], 70)



In [35]:
pred_dict = {"FVC": pred[:, 1], "Confidence": conf}
pred_df = pd.DataFrame(pred_dict)



In [36]:
sub["Confidence"] = pred_df["Confidence"].values
sub["FVC"] = pred_df["FVC"].values



In [37]:
subm = sub[["Patient_Week", "FVC", "Confidence"]].copy()



In [38]:
subm[300:400]



,Patient_Week,FVC,Confidence
300,ID00135637202224630271439_85,3535.024414,355.871094
301,ID00135637202224630271439_86,3535.243652,359.555908
302,ID00135637202224630271439_87,3536.945801,362.719727
303,ID00135637202224630271439_88,3538.416992,365.936279
304,ID00135637202224630271439_89,3540.874756,369.278809
...,...,...,...
395,ID00127637202219096738943_74,1510.516968,197.747681
396,ID00127637202219096738943_75,1505.935303,201.083496
397,ID00127637202219096738943_76,1501.193359,204.308350
398,ID00127637202219096738943_77,1496.925293,207.379883


In [39]:
subm.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", subm.shape)
print(subm.head())

Wrote submission.csv with shape: (1908, 3)
                   Patient_Week          FVC  Confidence
0  ID00126637202218610655908_-3  2517.398682  106.441162
1  ID00126637202218610655908_-2  2508.670898  110.839111
2  ID00126637202218610655908_-1  2500.551270  114.771973
3   ID00126637202218610655908_0  2492.060303  119.116943
4   ID00126637202218610655908_1  2483.788086  123.337402
